# Data-Dependent RoPE for State-Space Tracking

The Core Problem: Why SSMs Blur Token OrderingStandard Rotary Position Embeddings (RoPE) were designed for Transformers. They work by rotating the Key ($K$) and Query ($Q$) vectors in 2D space based on their absolute token position index ($t = 0, 1, 2, \dots$):$$\theta_t = t \cdot \omega$$This static rotation gives Transformers a precise sense of distance between tokens across full attention matrices.However, Selective State Space Models (Mamba, Linear-RNNs) do not construct $Q \times K^\top$ pairwise attention matrices. Instead, they compress token histories into a single hidden state vector ($h_t$) using an input-dependent decay factor ($\Delta_t$):$$h_t = \alpha_t h_{t-1} + B_t x_t$$This creates a severe structural weakness for state-tracking tasks:

Static RoPE (Transformers / Traditional SSMs):
  Token 1 ("{")      ---> Position 0 ---> Rotate by 0°
  Token 2 ("key")    ---> Position 1 ---> Rotate by 15°
  Token 3 (":")      ---> Position 2 ---> Rotate by 30°
  Token 4 ("val")    ---> Position 3 ---> Rotate by 45°
  (Rotation happens purely based on token count, regardless of content.)

Data-Dependent RoPE (Mamba-3 / Modern SSMs):
  Token 1 ("{")      ---> Structural Start   ---> Phase shift +90° (Open Context)
  Token 2 ("key")    ---> Identifiers        ---> Phase shift +0°  (Hold Context)
  Token 3 (":")      ---> Delimiter          ---> Phase shift +180° (Switch to Value)
  Token 4 ("val")    ---> Target Payload     ---> Phase shift +0°  (Read Value)
  (Rotation angle θ_t is dynamically calculated FROM THE TOKEN CONTENT x_t itself.)

What Breaks in Practice?When an SSM processes a structured document (e.g., JSON parsing, code AST execution, nested function calls, or state machines):Static position index $t$ carries no semantic information. Knowing that a } is at token index 409 doesn't tell the model which nested block it just closed—only how many total characters were processed.State Blurring: If two key-value pairs are separated by variable amounts of whitespace or irremediable filler text, a standard SSM's decay ($\alpha_t$) attenuates the key information before reaching the lookup point.

## Understanding Data dependent RoPE

The Compass & Journal AnalogyImagine you are walking through a giant maze (a long text document), and your goal is to remember where you are so you don't get lost.You are holding two tools:A Journal (The State / Memory Vector $h_t$)A Compass (The Rotary Position Embedding / RoPE)How Old Models (Standard RoPE) Use the CompassIn traditional models, your compass needle turns automatically based on your step count:Step 1: Turn needle to $10^\circ$ North.Step 2: Turn needle to $20^\circ$ North.Step 3: Turn needle to $30^\circ$ North.It doesn't matter what you pass on your walk.Whether Step 2 was a gold coin, a dead end, or a blank wall, the compass turns strictly based on the step number.


Why this breaks in long texts or code:
If you read a programming file or a complex JSON document, step #450 might be a } (closing a block) or it might just be a random space bar or word.
Counting steps tells you how many tokens you read, but it tells you nothing about what structural room you just entered. Memory gets blurred together.


How New Models (Data-Dependent RoPE) Use the CompassInstead of turning the compass based on a clock or step count, the compass needle turns based on WHAT YOU JUST SAW.The content of the token itself ($x_t$) physically twists the compass needle:You see a { (Start of a function): You twist the compass $90^\circ$ to the Right. (You are now pointing East = Level 1 Scope).You see filler text (x = 5): $0^\circ$ twist. (Stay pointing East—you're still inside Level 1).You see another ( (Nested math): You twist another $90^\circ$ Right. (You are now pointing South = Level 2 Scope).You see a ) (Close nested math): You twist $90^\circ$ Left. (You turn back to East = Level 1 Scope).Why is this a massive deal?It creates "Semantic Anchors":Instead of forgetting old information as distance increases, the model uses dynamic rotations to "lock" important ideas at specific angles.Angles don't destroy memory:If you shrink a number to make room for new text, you erase the past. But if you rotate a vector in 2D space, you change its direction without shrinking its size/importance.Content Drives Position:Position isn't just "Token #50". Position becomes "Inside the 2nd nested bracket, after the 'if' statement".


## 2. Standard Static RoPE (How Transformers & Basic Models Do It)

Static RoPE assigns rotation angles **purely based on token positions** ($0, 1, 2, 3, \dots$).

Let's say every position index adds $+15^\circ$ to the state vector:

| Token Step | Input Word | Position Index ($t$) | Rotation Angle ($\theta_t = t \times 15^\circ$) | What the Vector Does |
| --- | --- | --- | --- | --- |
| **0** | `def` | $t = 0$ | $0^\circ$ | Vector stays at $0^\circ$ |
| **1** | `foo` | $t = 1$ | $15^\circ$ | Rotates to $15^\circ$ |
| **2** | `(` | $t = 2$ | $30^\circ$ | Rotates to $30^\circ$ |
| **3** | `x` | $t = 3$ | $45^\circ$ | Rotates to $45^\circ$ |
| **4** | `)` | $t = 4$ | $60^\circ$ | Rotates to $60^\circ$ |
| **5** | `:` | $t = 5$ | $75^\circ$ | Rotates to $75^\circ$ |
| **6** | `return` | $t = 6$ | $90^\circ$ | Rotates to $90^\circ$ |
| **7** | `x` | $t = 7$ | $105^\circ$ | Rotates to $105^\circ$ |

### Why This Fails for Scope Tracking:

Look at the parameter `x` at position 3 (Angle $45^\circ$) and the return value `x` at position 7 (Angle $105^\circ$).

If someone adds **spaces or comments** in the code:


$$\texttt{"def foo(   x   ) : return x"}$$


Now the parameter `x` sits at position 6 (Angle $90^\circ$)!

The model’s internal vector for `x` shifted from $45^\circ$ to $90^\circ$ **just because of extra whitespace**. The angle tells the model *where in the sentence* it is, but **not what scope state it is in**.

---

## 3. Data-Dependent RoPE ($\theta_t = f(x_t)$)

Now, instead of reading position numbers, the linear layer $\mathbf{W}_\theta x_t$ looks at the **meaning of the current word** to calculate the rotation angle:

* **Keyword (`def`, `return`):** Angle shift $0^\circ$ (Keep current layer)
* **Identifier (`foo`, `x`):** Angle shift $0^\circ$ (Keep current layer)
* **Open Bracket `(`:** Angle shift **$+90^\circ$** (Jump into deeper scope)
* **Close Bracket `)`:** Angle shift **$-90^\circ$** (Drop back to main scope)

Let's watch what happens to the state's total angle as the sequence runs:

| Token | Meaning | Dynamic Angle Shift | Cumulative State Angle | What the Model "Understands" |
| --- | --- | --- | --- | --- |
| `def` | Function Definition | $+0^\circ$ | $0^\circ$ | Main level |
| `foo` | Function Name | $+0^\circ$ | $0^\circ$ | Main level |
| `(` | **Open Scope** | **$+90^\circ$** | **$90^\circ$** | **Shifted into INSIDE_ARGS scope** |
| `x` | Parameter | $+0^\circ$ | $90^\circ$ | Parameter `x` lives at $90^\circ$ vector angle |
| `)` | **Close Scope** | **$-90^\circ$** | **$0^\circ$** | **Returned to Main level** |
| `:` | Delimiter | $+0^\circ$ | $0^\circ$ | Main level |
| `return` | Return Statement | $+0^\circ$ | $0^\circ$ | Main level |
| `x` | Return Variable | $+0^\circ$ | $0^\circ$ | Return `x` lives at $0^\circ$ vector angle |

---

## 4. Why This Works

Look at the two `x` tokens now:

1. **Parameter `x` inside `( x )`:** Holds a total vector angle of **$90^\circ$**.
2. **Return value `x` after `return`:** Holds a total vector angle of **$0^\circ$**.

Even though both tokens are the exact same word (`"x"`), their internal state vectors are pointing in **completely orthogonal directions** ($90^\circ$ apart).

And if you add 50 spaces or comments:


$$\texttt{"def foo(       x       ) : return x"}$$

All the space tokens emit an angle shift of $0^\circ$. So parameter `x` **still sits at $90^\circ$**, and return `x` **still sits at $0^\circ$**.

The model tracked the **logical state of the code**, completely immune to distance, token counts, or filler text.

### Imagine Reading an Article About a Movie Plot

Suppose the text is:

> *"In the movie, **John** went to the store. After a long chase sequence, **he** bought an apple."*

---

### 1. Standard Static RoPE (Word-Count Clock)

Standard RoPE turns the angle dial purely based on **how many words away** tokens are:

* Word 1 (`John`): Dial at **$10^\circ$**
* Word 2–8 (`went...chase`): Dial ticks up $20^\circ, 30^\circ, 40^\circ \dots$
* Word 9 (`he`): Dial sits at **$90^\circ$**

#### The Problem:

Because the dial turned strictly by word count, the relationship between `John` and `he` is encoded as *"8 words apart."*

If an editor adds a bunch of descriptive filler:

> *"In the movie, **John** went to the store located on 5th avenue past the bridge while escaping a giant robot. After a long chase sequence, **he** bought an apple."*

Now `he` is **22 words away** (Dial at $220^\circ$). The model struggles because the distance angle changed completely, even though the **grammatical connection** between `John` and `he` is identical.

---

### 2. Data-Dependent RoPE (Meaning-Driven Dial)

Instead of a word counter, the current word itself **turns the dial dynamically**:

* Word 1 (`John`): **Subject introduced** $\rightarrow$ Twist dial to **$90^\circ$** (Anchor: *Male Protagonist State*).
* Filler words (`went...chase`): **Neutral text** $\rightarrow$ Twist dial **$0^\circ$** (Keep current state pointing at $90^\circ$).
* Word 9 (`he`): **Pronoun** $\rightarrow$ Checks current state ($90^\circ$) and immediately connects `he` back to `John`.

#### Why it succeeds:

Even if you insert 50 filler words, all those neutral filler words emit a $0^\circ$ twist.

When the model reaches `he`, the state is **still pointing directly at $90^\circ$**. It tracks that `he` refers to `John` based on **narrative role**, completely ignoring how many filler words sat in between.

In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Set seed for strict side-by-side fairness
torch.manual_seed(42)

# =====================================================================
# 1. DATASET GENERATOR (Scope Depth Tracking)
# =====================================================================

VOCAB = {
    "<PAD>": 0, "{": 1, "}": 2, "[": 3, "]": 4, 
    "(": 5, ")": 6, "x": 7, "y": 8, "+": 9
}

class ScopeDataset(Dataset):
    def __init__(self, num_samples=3000, max_len=48):
        self.samples = []
        self.labels = []
        
        open_brackets = {1, 3, 5}   # {, [, (
        fillers = [7, 8, 9]         # x, y, +
        close_map = {1: 2, 3: 4, 5: 6}

        for _ in range(num_samples):
            seq_len = torch.randint(16, max_len, (1,)).item()
            seq, targets, stack = [], [], []
            current_depth = 0

            for _ in range(seq_len):
                p = torch.rand(1).item()
                if p < 0.35 and len(stack) < 8:  # Open scope
                    b = list(open_brackets)[torch.randint(0, 3, (1,)).item()]
                    stack.append(b)
                    current_depth += 1
                    seq.append(b)
                elif p < 0.70 and len(stack) > 0: # Close scope
                    last_open = stack.pop()
                    seq.append(close_map[last_open])
                    targets.append(current_depth) # Capture depth at bracket
                    current_depth -= 1
                    continue
                else:                             # Filler token
                    seq.append(fillers[torch.randint(0, 3, (1,)).item()])
                
                targets.append(current_depth)
                
            self.samples.append(torch.tensor(seq, dtype=torch.long))
            self.labels.append(torch.tensor(targets, dtype=torch.long))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx], self.labels[idx]

def collate_fn(batch):
    inputs, targets = zip(*batch)
    inputs_padded = nn.utils.rnn.pad_sequence(inputs, batch_first=True, padding_value=0)
    targets_padded = nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=-100)
    return inputs_padded, targets_padded


# =====================================================================
# 2. MODULE A: STANDARD STATIC RoPE (Position-Based)
# =====================================================================

class StaticRoPE(nn.Module):
    """ Standard RoPE: Rotates vectors based purely on sequence index (t = 0, 1, 2, ...) """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # Frequency scale base
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x):
        B, T, D = x.shape
        pos = torch.arange(T, device=x.device).type_as(self.inv_freq) # [T]
        angles = torch.einsum("i,j->ij", pos, self.inv_freq)          # [T, D/2]
        angles = angles.unsqueeze(0).repeat(B, 1, 1)                 # [B, T, D/2]

        x1 = x[..., 0::2]
        x2 = x[..., 1::2]

        cos = torch.cos(angles)
        sin = torch.sin(angles)

        x1_rot = x1 * cos - x2 * sin
        x2_rot = x1 * sin + x2 * cos

        return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)


# =====================================================================
# 3. MODULE B: DATA-DEPENDENT RoPE (Content-Based)
# =====================================================================

class DataDependentRoPE(nn.Module):
    """ Data-Dependent RoPE: Rotates vectors dynamically from x_t content """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.angle_proj = nn.Linear(d_model, d_model // 2)

    def forward(self, x):
        angles = self.angle_proj(x) # [B, T, D/2]

        x1 = x[..., 0::2]
        x2 = x[..., 1::2]

        cos = torch.cos(angles)
        sin = torch.sin(angles)

        x1_rot = x1 * cos - x2 * sin
        x2_rot = x1 * sin + x2 * cos

        return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)


# =====================================================================
# 4. GENERAL SSM ARCHITECTURE
# =====================================================================

class ModularSSM(nn.Module):
    def __init__(self, rope_type="data_dependent", vocab_size=10, d_model=32, d_state=16, num_classes=10):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        
        if rope_type == "data_dependent":
            self.rope = DataDependentRoPE(d_model)
        elif rope_type == "static":
            self.rope = StaticRoPE(d_model)
        else:
            raise ValueError("Invalid rope_type")

        self.A_log = nn.Parameter(
            torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_model, 1))
        )
        self.x_proj = nn.Linear(d_model, d_state + d_model, bias=False)
        self.dt_proj = nn.Linear(d_model, d_model)

        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids):
        B, T = input_ids.shape
        x_emb = self.embedding(input_ids)

        x_rotated = self.rope(x_emb)

        h_state = torch.zeros(B, self.d_model, self.d_state, device=input_ids.device)
        outputs = []
        A = -torch.exp(self.A_log)

        for t in range(T):
            x_t = x_rotated[:, t, :]

            proj = self.x_proj(x_t)
            B_t, dt_raw = torch.split(proj, [self.d_state, self.d_model], dim=-1)
            delta = F.softplus(self.dt_proj(dt_raw))

            A_bar = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0))
            B_bar = delta.unsqueeze(-1) * B_t.unsqueeze(1)

            # Read step including current x_t representation
            readout_t = torch.sum(h_state, dim=-1) + x_t
            outputs.append(readout_t)

            # State update
            h_state = A_bar * h_state + B_bar * x_t.unsqueeze(-1)

        sequence_out = torch.stack(outputs, dim=1)
        logits = self.classifier(sequence_out)
        return logits


# =====================================================================
# 5. BENCHMARK COMPARISON PIPELINE
# =====================================================================

def train_and_eval(model, train_loader, test_loader, epochs=50, device="cpu"):
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    
    accuracies = []
    
    for epoch in range(1, epochs + 1):
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits.view(-1, 10), targets.view(-1))
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                logits = model(inputs)
                preds = torch.argmax(logits, dim=-1)
                mask = (targets != -100)
                correct += (preds[mask] == targets[mask]).sum().item()
                total += mask.sum().item()

        acc = (correct / total) * 100
        accuracies.append(acc)
    return accuracies

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running Benchmark on Device: {device}\n")

    train_loader = DataLoader(ScopeDataset(num_samples=3000), batch_size=32, shuffle=True, collate_fn=collate_fn)
    test_loader = DataLoader(ScopeDataset(num_samples=500), batch_size=32, shuffle=False, collate_fn=collate_fn)

    # 1. Instantiate Models
    torch.manual_seed(42)
    model_static = ModularSSM(rope_type="static").to(device)
    
    torch.manual_seed(42)
    model_data_dep = ModularSSM(rope_type="data_dependent").to(device)

    print("--- Training Model A: Standard Static RoPE ---")
    static_accs = train_and_eval(model_static, train_loader, test_loader, epochs=50, device=device)

    print("--- Training Model B: Data-Dependent RoPE ---")
    data_dep_accs = train_and_eval(model_data_dep, train_loader, test_loader, epochs=50, device=device)

    print("\n" + "="*55)
    print(f"{'Epoch':<6} | {'Static RoPE Acc (%)':<20} | {'Data-Dep RoPE Acc (%)':<20}")
    print("="*55)
    for i in range(10):
        print(f"{i+1:<6} | {static_accs[i]:<20.2f} | {data_dep_accs[i]:<20.2f}")
    print("="*55)

    # Qualitative Test Comparison
    sample_str = "{ x + [ y ( x ) ] }"
    tokens = [VOCAB[c] for c in sample_str.split()]
    input_tensor = torch.tensor([tokens], dtype=torch.long).to(device)

    model_static.eval()
    model_data_dep.eval()
    with torch.no_grad():
        pred_static = torch.argmax(model_static(input_tensor), dim=-1).squeeze(0).cpu().tolist()
        pred_data_dep = torch.argmax(model_data_dep(input_tensor), dim=-1).squeeze(0).cpu().tolist()

    print("\n--- QUALITATIVE TEST INFERENCE ---")
    print(f"Tokens:            {'  '.join(sample_str.split())}")
    print(f"Static RoPE Preds: {'  '.join(map(str, pred_static))}")
    print(f"Data-Dep RoPE Pred:{'  '.join(map(str, pred_data_dep))}")

if __name__ == "__main__":
    main()

Running Benchmark on Device: cpu

--- Training Model A: Standard Static RoPE ---
--- Training Model B: Data-Dependent RoPE ---

Epoch  | Static RoPE Acc (%)  | Data-Dep RoPE Acc (%)
1      | 43.99                | 43.24               
2      | 54.03                | 50.69               
3      | 60.75                | 61.09               
4      | 60.97                | 68.51               
5      | 66.22                | 75.49               
6      | 70.94                | 66.23               
7      | 76.62                | 79.27               
8      | 79.70                | 84.40               
9      | 81.12                | 81.95               
10     | 73.77                | 89.03               

--- QUALITATIVE TEST INFERENCE ---
Tokens:            {  x  +  [  y  (  x  )  ]  }
Static RoPE Preds: 1  1  1  2  2  3  3  3  2  1
Data-Dep RoPE Pred:1  1  1  2  2  3  3  3  2  1
